# Problem 6: From Gradient Descent to Adam
SRH Berlin, Algorithms and Data Structures. Group: Manas Dogra (100006131), Monish Chopra (100007201), Aaditya Dawer (100006382)

Every optimizer has one method, `step(w, g)`: it takes the position `w` and the slope `g`, and gives the new position.
Run it with Runtime, then Run all.

In [ ]:
import time, tracemalloc
import numpy as np
import matplotlib.pyplot as plt

## 1. The optimizers
Extra memory: GD 0, Momentum 1d, RMSProp 1d, Adam 2d (d = number of parameters).

In [ ]:
class GD:                       # Stages 1 and 2: w = w - eta * g
    def __init__(self, eta):
        self.eta = eta
    def step(self, w, g):
        return w - self.eta * g

MiniBatchGD = GD                # same update, only g comes from a small batch

class Momentum:                 # Stage 3: running average of g
    def __init__(self, eta, beta1=0.9):
        self.eta, self.beta1, self.v = eta, beta1, 0
    def step(self, w, g):
        self.v = self.beta1 * self.v + (1 - self.beta1) * g
        return w - self.eta * self.v

class RMSProp:                  # Stage 4: running average of g squared
    def __init__(self, eta, beta2=0.9, eps=1e-8):
        self.eta, self.beta2, self.eps, self.s = eta, beta2, eps, 0
    def step(self, w, g):
        self.s = self.beta2 * self.s + (1 - self.beta2) * g ** 2
        return w - self.eta * g / (np.sqrt(self.s) + self.eps)

class Adam:                     # Stage 5: Momentum + RMSProp + bias correction
    def __init__(self, eta, beta1=0.9, beta2=0.9, eps=1e-8):
        self.eta, self.beta1, self.beta2, self.eps = eta, beta1, beta2, eps
        self.m, self.s, self.t = 0, 0, 0
    def step(self, w, g):
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * g
        self.s = self.beta2 * self.s + (1 - self.beta2) * g ** 2
        m_hat = self.m / (1 - self.beta1 ** self.t)    # bias correction
        s_hat = self.s / (1 - self.beta2 ** self.t)    # bias correction
        return w - self.eta * m_hat / (np.sqrt(s_hat) + self.eps)

class Newton:                   # w = w - H^-1 g (needs the d x d matrix H)
    def __init__(self, max_step=None):
        self.max_step = max_step
    def step(self, w, g, H):
        move = np.linalg.solve(H, g)
        if g @ move <= 0:                       # not downhill, use the plain slope
            move = g
        size = np.linalg.norm(move)
        if self.max_step and size > self.max_step:
            move = move * self.max_step / size  # do not jump too far
        return w - move

## 2. Tests with our hand answers (Q1 to Q5)

In [ ]:
def g_hand(w):                  # slope of f = 0.5*w1^2 + 5*w2^2
    return np.array([w[0], 10 * w[1]])

w0 = np.array([4.0, 1.0])

o = GD(0.18); w1 = o.step(w0, g_hand(w0)); w2 = o.step(w1, g_hand(w1))
assert np.allclose(w1, [3.28, -0.8]) and np.allclose(w2, [2.6896, 0.64])              # Q1

o = Momentum(0.18, 0.5); w1 = o.step(w0, g_hand(w0)); w2 = o.step(w1, g_hand(w1))
assert np.allclose(w1, [3.64, 0.1]) and np.allclose(w2, [3.1324, -0.44])              # Q2

o = RMSProp(0.5, 0.9, 0.0); w1 = o.step(w0, g_hand(w0))
assert np.allclose(o.s, [1.6, 10]) and np.allclose(w1, [2.4189, -0.5811], atol=1e-4)  # Q3 RMSProp

o = Adam(0.5, 0.9, 0.9, 0.0); w1 = o.step(w0, g_hand(w0))
m_hat, s_hat = o.m / 0.1, o.s / 0.1
assert np.allclose(m_hat, [4, 10]) and np.allclose(s_hat, [16, 100])
assert np.allclose(w1, [3.5, 0.5]) and np.allclose(m_hat / np.sqrt(s_hat), [1, 1])    # Q3 Adam

m1 = 0.01 * 10
assert np.isclose(m1, 0.1) and np.isclose(m1 / 0.01, 10)                              # Q4

assert np.allclose(Newton().step(w0, g_hand(w0), np.diag([1.0, 10.0])), [0, 0])      # Q5
print("All tests passed")

## 3. Experiment 1: paths on two landscapes (start at (-1.5, 2), 200 steps)

In [ ]:
canyon = dict(f=lambda w: 0.5 * w[0]**2 + 25 * w[1]**2,
              grad=lambda w: np.array([w[0], 50 * w[1]]),
              hess=lambda w: np.diag([1.0, 50.0]), goal=(0, 0), ylim=(-2, 2.5),
              opts=[GD(0.035), Momentum(0.035), RMSProp(0.05), Adam(0.1, 0.9, 0.999), Newton()])
rosen = dict(f=lambda w: (1 - w[0])**2 + 100 * (w[1] - w[0]**2)**2,
             grad=lambda w: np.array([-2 * (1 - w[0]) - 400 * w[0] * (w[1] - w[0]**2), 200 * (w[1] - w[0]**2)]),
             hess=lambda w: np.array([[2 - 400 * w[1] + 1200 * w[0]**2, -400 * w[0]], [-400 * w[0], 200.0]]),
             goal=(1, 1), ylim=(-1, 3),
             opts=[GD(0.0005), Momentum(0.0005), RMSProp(0.02), Adam(0.05), Newton(max_step=1.0)])

def run(opt, p, T=200):
    w = np.array([-1.5, 2.0])
    path, values = [w], [p["f"](w)]
    for _ in range(T):
        if isinstance(opt, Newton):
            w = opt.step(w, p["grad"](w), p["hess"](w))
        else:
            w = opt.step(w, p["grad"](w))
        path.append(w)
        values.append(p["f"](w))
    return np.array(path), np.array(values)

names = ["Gradient Descent", "Momentum", "RMSProp", "Adam", "Newton"]
colors = ["tab:blue", "tab:green", "tab:orange", "tab:red", "tab:purple"]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for row, (title, p) in enumerate([("Canyon", canyon), ("Rosenbrock", rosen)]):
    xs, ys = np.linspace(-2, 2, 200), np.linspace(*p["ylim"], 200)
    Z = np.array([[p["f"](np.array([a, b])) for a in xs] for b in ys])
    axes[row, 0].contour(xs, ys, np.log10(Z + 1e-12), levels=25, cmap="Greys", alpha=0.6)
    for opt, name, color in zip(p["opts"], names, colors):
        path, values = run(opt, p)
        axes[row, 0].plot(path[:, 0], path[:, 1], "-o", ms=2, color=color, label=name)
        axes[row, 1].plot(np.log10(values + 1e-16), color=color, label=name)
    axes[row, 0].plot(*p["goal"], "k*", ms=14, label="minimum")
    axes[row, 0].set_ylim(*p["ylim"]); axes[row, 0].set_title(title + ": paths")
    axes[row, 1].set_title(title + ": log10 f(w) against step"); axes[row, 1].set_xlabel("step")
    axes[row, 0].legend(fontsize=8); axes[row, 1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("experiment1_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

What we see: on the canyon Newton needs 1 step, Gradient Descent zig-zags, Momentum swings from side to side, Adam and RMSProp go in a smoother curve. On Rosenbrock only Adam (and Newton) get close to (1, 1) in 200 steps. The others barely leave the start. The flat line at -16 only means the loss is below 1e-16.

## 4. Experiment 2: linear regression with 50,000 samples
The columns of `X` have very different sizes (0.1 to 10), a bit like the canyon.

In [ ]:
rng = np.random.default_rng(0)
N, d, b = 50_000, 50, 64
X = rng.standard_normal((N, d)) * np.logspace(-1, 1, d)
y = X @ rng.standard_normal(d) + 0.1 * rng.standard_normal(N)

def loss(w):
    return 0.5 * np.mean((X @ w - y) ** 2)

def slope(w, batch):            # batch = None means use all N rows
    rows = slice(None) if batch is None else rng.integers(0, N, size=batch)
    return X[rows].T @ (X[rows] @ w - y[rows]) / len(y[rows])

def train(opt, batch, steps, check_every):
    w, clock, times, losses = np.zeros(d), 0.0, [0.0], [loss(np.zeros(d))]
    for t in range(1, steps + 1):
        t0 = time.perf_counter()
        w = opt.step(w, slope(w, batch))
        clock += time.perf_counter() - t0          # only the update steps are timed
        if t % check_every == 0:
            times.append(clock); losses.append(loss(w))
    return times, losses

def peak_memory_kb(make, batch):
    opt, w = make(), np.zeros(d)
    tracemalloc.start()
    for _ in range(20):
        w = opt.step(w, slope(w, batch))
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak / 1024

runs = {"Full-batch GD": (lambda: GD(0.015), None, 300, 10),
        "Mini-batch GD": (lambda: GD(0.01), b, 6000, 100),
        "Momentum": (lambda: Momentum(0.01), b, 6000, 100),
        "RMSProp": (lambda: RMSProp(0.005), b, 6000, 100),
        "Adam": (lambda: Adam(0.02, 0.9, 0.999), b, 6000, 100)}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
memory = []
for name, (make, batch, steps, every) in runs.items():
    times, losses = train(make(), batch, steps, every)
    memory.append(peak_memory_kb(make, batch))
    axes[0].plot(times, losses, label=name)
    print(f"{name:14s} final loss = {losses[-1]:.4f}   time = {times[-1]:.2f} s   memory = {memory[-1]:.0f} KB")
axes[0].set_yscale("log"); axes[0].set_xlabel("time (seconds)"); axes[0].set_ylabel("loss")
axes[0].set_title("Loss against time"); axes[0].legend()
axes[1].bar(list(runs), memory, color=["tab:blue", "tab:cyan", "tab:green", "tab:orange", "tab:red"])
axes[1].set_ylabel("peak memory (KB)"); axes[1].set_title("Peak memory"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig("experiment2_loss_time_memory.png", dpi=150, bbox_inches="tight")
plt.show()

What we see: full-batch GD is slow because every step uses all 50,000 rows (loss still about 0.3 after 0.7 seconds). The mini-batch methods reach about 0.01 in 0.1 seconds. Adam gets closest to the noise level (about 0.005). Full-batch needs the most memory because it uses all N rows in each step. The extra arrays of Momentum, RMSProp and Adam (400 to 800 bytes) are tiny. Times change from computer to computer.

## 5. Experiment 3: time per step as N grows

In [ ]:
sizes = [1_000, 3_000, 10_000, 30_000, 100_000, 200_000]
full, mini = [], []
for n in sizes:
    Xn = rng.standard_normal((n, d)); yn = Xn[:, 0]; w = np.zeros(d)
    t0 = time.perf_counter()
    for _ in range(30):
        Xn.T @ (Xn @ w - yn) / n
    full.append((time.perf_counter() - t0) / 30)
    t0 = time.perf_counter()
    for _ in range(30):
        r = rng.integers(0, n, size=b); Xn[r].T @ (Xn[r] @ w - yn[r]) / b
    mini.append((time.perf_counter() - t0) / 30)
plt.figure(figsize=(7, 5))
plt.loglog(sizes, np.array(full) * 1000, "o-", label="Full-batch (about N*d)")
plt.loglog(sizes, np.array(mini) * 1000, "s-", label="Mini-batch (about b*d)")
plt.xlabel("N"); plt.ylabel("time per step (ms)"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.savefig("experiment3_time_vs_N.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary (Q6)
| Method | Time per step | Extra memory |
|---|---|---|
| Batch GD | N*d | 0 |
| Mini-batch GD | b*d | 0 |
| Momentum | b*d | 1d |
| RMSProp | b*d | 1d |
| Adam | b*d | 2d |
| Newton | N*d + d^3 | d^2 |